# Isaac Sim on Kaggle: Free T4 Robotics Simulation

**NVIDIA's robotics simulator, running inside a Kaggle notebook, on the free GPU quota. Plus an honest answer to whether you actually should.**

---

> Part of an open series on running **NVIDIA Isaac Sim on free GPUs**.
> Isaac Sim is free software (Apache 2.0) — only compute ever costs money,
> and this series is about not paying for that either.
>
> If this is useful, an upvote helps other people find it. Questions in the
> comments get answered.

---

## Why this notebook exists

NVIDIA **Isaac Sim** is the industry-standard robotics simulator — physics,
photorealistic RTX rendering, synthetic data generation, and RL environments
via Isaac Lab. It is also free software (Apache 2.0).

What is *not* free is the hardware people assume you need. Almost every guide
starts with "get a workstation with an RTX 4090" or "spin up an AWS g6e
instance." Both are real costs, and both are avoidable for learning.

Kaggle hands out **30 GPU hours a week on 2x T4** to anyone with a verified
phone number. This notebook asks a simple question: *is that enough to run
Isaac Sim?*

The short answer is **yes, with real caveats**, and by the end you will have
run a physics simulation and rendered frames from it without spending
anything. The caveats matter more than the yes, so I have not buried them.

### What you need before running this
1. **Accelerator: GPU T4 x2** (Settings panel, right-hand side)
2. **Internet: ON** — Isaac Sim pulls assets from NVIDIA at runtime and pip
   needs NVIDIA's package index. Without this the notebook cannot work.

## Step 1 — Preflight

Isaac Sim on a notebook GPU has four independent ways to fail: wrong GPU
class, old GLIBC, insufficient disk, blocked network. The installer's error
messages for these are famously unhelpful, and each failure costs you ~20
minutes of install time before it surfaces.

So: check all four in five seconds, before installing anything.

In [ ]:
import os, platform, shutil, subprocess, sys

def sh(cmd):
    try:
        return subprocess.run(cmd, shell=True, capture_output=True,
                              text=True, timeout=60).stdout.strip()
    except Exception:
        return ""

print("=" * 60)
ok = True

# --- GPU class -------------------------------------------------------------
# Turing (7.5) is the OLDEST architecture with RT cores. The T4 is 7.5, so we
# are exactly at the floor. Note what is NOT on the good list further down.
gpu = sh("nvidia-smi --query-gpu=name,memory.total,compute_cap "
         "--format=csv,noheader")
if not gpu:
    print("[FAIL] No GPU. Settings -> Accelerator -> GPU T4 x2")
    ok = False
else:
    for line in gpu.splitlines():
        name, mem, cap = [p.strip() for p in line.split(",")]
        cc = tuple(int(x) for x in cap.split("."))
        flag = "OK  " if cc >= (7, 5) and cc not in {(8, 0), (9, 0)} else "WARN"
        print(f"[{flag}] {name}, {mem}, compute {cap}")

# --- GLIBC -----------------------------------------------------------------
glibc = platform.libc_ver()[1]
gl = tuple(int(x) for x in glibc.split(".")[:2]) if glibc else (0, 0)
print(f"[{'OK  ' if gl >= (2,34) else 'FAIL'}] GLIBC {glibc} (need >= 2.34)")
ok &= gl >= (2, 34)

# --- Disk ------------------------------------------------------------------
# /kaggle/working is capped around 20GB; the isaacsim wheels alone are ~20GB.
# Install somewhere roomier or you will die at 95%.
for path in ["/kaggle/working", "/kaggle/temp", "/tmp"]:
    if os.path.isdir(path):
        free = shutil.disk_usage(path).free / 1e9
        print(f"[{'OK  ' if free > 25 else 'WARN'}] {free:6.1f} GB free at {path}")

# --- Network ---------------------------------------------------------------
net = sh("curl -s -o /dev/null -w '%{http_code}' --max-time 10 "
         "https://pypi.nvidia.com") == "200"
print(f"[{'OK  ' if net else 'FAIL'}] pypi.nvidia.com reachable"
      f"{'' if net else '  <- turn Internet ON in notebook settings'}")
ok &= net

print("=" * 60)
print("VERDICT:", "GO" if ok else "NO-GO — fix the FAILs above first")

### The RT core trap

This is worth stopping on, because it is the single most counterintuitive
thing about running Isaac Sim on rented or donated GPUs.

Isaac Sim's renderer needs **RT cores** — dedicated ray-tracing hardware.

| GPU | Compute cap | RT cores? | Isaac Sim |
|---|---|---|---|
| T4 (Kaggle/Colab free) | 7.5 | ✅ Yes | Works |
| L4 / L40S | 8.9 | ✅ Yes | Ideal |
| RTX 3090 / 4090 | 8.6 / 8.9 | ✅ Yes | Ideal |
| **A100** | **8.0** | ❌ **No** | Unsupported / crawls |
| **H100** | **9.0** | ❌ **No** | Unsupported / crawls |

So the $2/hr A100 you were about to rent is *worse for this job* than
Kaggle's free T4. Datacenter compute parts strip the RT cores out because
their customers are training transformers, not rendering. If you take one
thing from this notebook, take this one — it saves real money.

### The Python version trap

This one cost me a run, so it is worth stating plainly: **the `isaacsim` wheels
are pinned to exact Python versions**, and pip's error message when you get it
wrong is genuinely unhelpful — it says "no matching distribution" without ever
mentioning Python.

| Isaac Sim | Requires Python | Works on Kaggle (3.12)? |
|---|---|---|
| 4.0 – 4.5 | **3.10 only** | ❌ No |
| 5.0 – 5.1 | **3.11 only** | ❌ No |
| **6.0.x** | **3.12** | ✅ Yes |

Nearly every Isaac Sim tutorial online pins `==4.5.0`, because that was current
when they were written. On a modern Python that install cannot succeed. If you
copy a pinned version out of a blog post and see:

```
ERROR: Could not find a version that satisfies the requirement isaacsim==4.5.0
       (from versions: 6.0.0.0, 6.0.0.1, 6.0.1.0)
```

...that "from versions" list is pip telling you which builds match *your*
interpreter. Take the newest one rather than hunting for the pinned one.

The cell below checks this before installing, so the failure is a clear
message instead of a twenty-minute wait ending in a stack trace.

In [ ]:
import sys
py = sys.version_info
print(f"notebook Python: {py.major}.{py.minor}.{py.micro}")

ISAAC_FOR_PY = {(3, 10): "4.5.0", (3, 11): "5.1.0", (3, 12): "6.0.1.0"}
version = ISAAC_FOR_PY.get((py.major, py.minor))

if version:
    print(f"-> use isaacsim=={version}")
else:
    print(f"-> no known isaacsim build for Python {py.major}.{py.minor}")
    print("   check available builds with:")
    print("   pip index versions isaacsim --extra-index-url https://pypi.nvidia.com")

## Step 2 — Install

About 20GB and 15–20 minutes. Two environment variables are non-obvious and
mandatory: Isaac Sim refuses to start headless without acknowledging the EULA
and the telemetry prompt, and it does not tell you clearly that this is why.

In [ ]:
# Headless Isaac Sim will not launch without these acknowledgements.
os.environ["ACCEPT_EULA"] = "Y"
os.environ["OMNI_KIT_ACCEPT_EULA"] = "YES"
os.environ["PRIVACY_CONSENT"] = "Y"

# Keep the multi-GB shader cache off the size-capped working directory.
CACHE = "/kaggle/temp/omni_cache" if os.path.isdir("/kaggle/temp") else "/tmp/omni_cache"
os.makedirs(CACHE, exist_ok=True)
os.environ["OMNI_CACHE_ROOT"] = CACHE

In [ ]:
%%time
# ~20GB. Grab a coffee. If this dies partway, do NOT try to repair it --
# restart the session and run once more. In-place recovery basically never
# works and costs more time than starting clean.
!pip install -q "isaacsim[all,extscache]=={version}" --extra-index-url https://pypi.nvidia.com

## Step 3 — Vulkan, the wall nobody warns you about

Here is where a straightforward install stops being straightforward.

Isaac Sim renders through **Vulkan**. Kaggle's container ships the CUDA stack
but **not the NVIDIA Vulkan ICD file** — the small JSON that tells the Vulkan
loader which driver library to use. Without it you get:

```
[Error] [omni.rtx] VkResult: ERROR_INCOMPATIBLE_DRIVER
[Error] [omni.rtx] vkCreateInstance failed. Vulkan 1.1 is not supported
```

...and then the kernel dies outright, which is a spectacularly unhelpful way
to learn that a JSON file is missing.

The message blames the driver. The driver is fine — the T4 has RT cores and a
working NVIDIA driver. What is missing is the *registration*. So we install
the Vulkan loader and write the ICD file ourselves.

In [ ]:
import glob, json, os, subprocess

def sh(c):
    r = subprocess.run(c, shell=True, capture_output=True, text=True)
    return (r.stdout + r.stderr).strip()

print(sh("nvidia-smi --query-gpu=driver_version --format=csv,noheader"))

# 1. The Vulkan loader itself.
sh("apt-get update -qq && apt-get install -y -qq libvulkan1 vulkan-tools")

# 2. Find the NVIDIA GLX library the ICD must point at.
cands = (glob.glob("/usr/lib/x86_64-linux-gnu/libGLX_nvidia.so*")
         + glob.glob("/usr/lib/libGLX_nvidia.so*")
         + glob.glob("/usr/local/nvidia/lib64/libGLX_nvidia.so*"))
print("libGLX_nvidia candidates:", cands or "NONE FOUND")

# 3. Register it. This is the file whose absence causes the error above.
os.makedirs("/usr/share/vulkan/icd.d", exist_ok=True)
icd = {"file_format_version": "1.0.0",
       "ICD": {"library_path": cands[0] if cands else "libGLX_nvidia.so.0",
               "api_version": "1.3.242"}}
with open("/usr/share/vulkan/icd.d/nvidia_icd.json", "w") as f:
    json.dump(icd, f, indent=2)
print("wrote nvidia_icd.json ->", icd["ICD"]["library_path"])

# 4. Did it take?
info = sh("vulkaninfo --summary")
ok = "deviceName" in info or "GPU id" in info
print("\nVulkan sees a device:", ok)
print(info[:900] if info else "(vulkaninfo produced no output)")

## Step 4 — First light

The moment of truth. `SimulationApp` must be constructed **before** any other
Isaac import — the app bootstraps the extension system that every subsequent
`isaacsim.*` / `omni.*` import depends on. Importing them first is the most
common beginner error and produces a confusing `ModuleNotFoundError`.

One more version note: **Isaac Sim 4.5 renamed the extensions** from
`omni.isaac.*` to `isaacsim.*`. Tutorials written for 4.2 and earlier use the
old paths. The code below tries the new layout first and falls back, so it
works either way.

In [ ]:
from isaacsim import SimulationApp

# MUST come first, before any other isaacsim/omni import.
# Kaggle allocates TWO T4s. Isaac Sim's PhysX CUDA context manager fails to
# initialise when it has to pick among multiple devices in this container
# ("Failed to create Cuda Context Manager" / "Unable to create
# PxCudaContextManager"), so pin the process to one GPU before ANY CUDA
# initialisation happens.
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ.setdefault("OMNI_KIT_ALLOW_ROOT", "1")

# Kit reads arbitrary settings off the command line, so extra flags go into
# sys.argv BEFORE SimulationApp is constructed. Drivers above 535.255 report
# their version in a way Kit misreads as incompatible; skipping the check is
# the documented workaround and is a no-op when the driver was already fine.
import sys
sys.argv += [
    "--/rtx/verifyDriverVersion/enabled=false",
    "--/physics/cudaDevice=0",       # match CUDA_VISIBLE_DEVICES above
]

simulation_app = SimulationApp({
    "headless": True,           # no display exists in a notebook
    "renderer": "RayTracedLighting",
    "width": 640, "height": 480,
})
print("SimulationApp up")

In [ ]:
# Isaac Sim 4.5 renamed omni.isaac.* -> isaacsim.*. Support both.
try:
    from isaacsim.core.api import World
    from isaacsim.core.api.objects import DynamicCuboid
    API = "4.5+ (isaacsim.*)"
except ImportError:
    from omni.isaac.core import World
    from omni.isaac.core.objects import DynamicCuboid
    API = "<=4.2 (omni.isaac.*)"

import numpy as np
print("using API layout:", API)

world = World(stage_units_in_meters=1.0)
world.scene.add_default_ground_plane()

cube = world.scene.add(DynamicCuboid(
    prim_path="/World/cube", name="cube",
    position=np.array([0.0, 0.0, 1.0]),
    size=0.2, color=np.array([0.9, 0.2, 0.1]),
))
world.reset()
print("scene ready")

## Step 5 — Does the physics actually work?

A dropped cube is the "hello world" of physics simulation, and it is a real
test: we can check the trajectory against the closed-form solution for free
fall. If PhysX is running correctly on the GPU, the numbers agree.

In [ ]:
import numpy as np

dt = 1.0 / 60.0
heights, times = [], []

for i in range(120):                     # 2 seconds at 60 Hz
    world.step(render=False)             # render=False -> physics only, fast
    pos, _ = cube.get_world_pose()
    heights.append(float(pos[2]))
    times.append(i * dt)

heights = np.array(heights); times = np.array(times)

# Compare the free-fall portion against z = z0 - 0.5*g*t^2
z0 = heights[0]
free_fall = heights > 0.11               # cube half-extent = 0.1
t_f, z_f = times[free_fall], heights[free_fall]
analytic = z0 - 0.5 * 9.81 * t_f ** 2
err = np.abs(z_f - analytic).max()

print(f"start height   : {z0:.4f} m")
print(f"rest height    : {heights[-1]:.4f} m  (expected ~0.10 = half extent)")
print(f"max deviation from analytic free fall: {err*1000:.2f} mm")
print("PhysX sane:", "YES" if err < 0.01 else "NO — investigate")

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.plot(times, heights, lw=2.5, label="Isaac Sim (PhysX)")
ax.plot(t_f, analytic, "--", lw=1.8, color="crimson",
        label=r"analytic  $z_0 - \frac{1}{2}gt^2$")
ax.axhline(0.1, color="gray", ls=":", lw=1, label="resting height")
ax.set_xlabel("time (s)"); ax.set_ylabel("cube height (m)")
ax.set_title("Free fall and impact, simulated on a free Kaggle T4")
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

The simulated curve tracks the analytic parabola until impact, then settles at
the cube's half-extent. That is a correct physics solve — on a free GPU.

## Step 6 — Rendering

Physics is the easy half. Rendering is what actually needs those RT cores, and
it is where the T4 earns its keep or doesn't.

In [ ]:
try:
    from isaacsim.sensors.camera import Camera
except ImportError:
    from omni.isaac.sensor import Camera

camera = Camera(
    prim_path="/World/camera",
    position=np.array([2.5, 2.5, 1.6]),
    frequency=20,
    resolution=(640, 480),
)
camera.initialize()
camera.set_world_pose(np.array([2.5, 2.5, 1.6]),
                      camera_axes="world")

# The first rendered frame triggers shader compilation and can take MINUTES
# with no progress output. This is normal. Do not interrupt it.
for _ in range(5):
    world.step(render=True)

rgb = camera.get_rgba()
print("frame:", None if rgb is None else rgb.shape)

In [ ]:
if rgb is not None:
    plt.figure(figsize=(8, 6))
    plt.imshow(rgb[:, :, :3])
    plt.axis("off")
    plt.title("RTX-rendered frame from a free Kaggle T4")
    plt.show()
else:
    print("No frame returned — usually means shader compilation is still "
          "running. Step the world a few more times and retry.")

## Step 7 — Clean shutdown

`SimulationApp.close()` matters more than usual here. Isaac Sim holds GPU
memory aggressively, and a notebook that does not close it will fail on the
next cell with an opaque out-of-memory error rather than anything informative.

In [ ]:
simulation_app.close()
print("closed cleanly")

## The honest verdict

It works. You should probably still not do your real work this way.

**What Kaggle is genuinely good for here:**
- Learning Isaac Sim's API without buying hardware
- Physics-only work (`render=False`), which is fast and cheap
- Running someone else's reproducible robotics notebook — including this one

**Where it hurts:**
- **The 20GB install repeats every session.** Kaggle wipes the disk. Twenty
  minutes of every session, gone, forever.
- **Shader cache dies with it.** First-frame render costs minutes, every time.
- **12-hour session cap**, so no long training runs.
- **T4 is the floor**, not a comfortable margin. 16GB VRAM constrains scene
  complexity considerably.

### The pattern that actually works

Split generation from publication:

> **Generate on a persistent box → publish the artifacts to Kaggle.**

Do heavy Isaac Sim work somewhere with a disk that survives — Lightning AI's
free tier gives 80 GPU hours a month *with persistent storage*, so you install
once. Upload the outputs as a Kaggle Dataset. Kaggle notebooks then consume
pre-generated data and only need a GPU for the light work, which starts in
seconds.

That is what the rest of this series does, and it is why the next notebook —
training a vision model on Isaac Sim synthetic data — runs here in under two
minutes instead of twenty.

**Found this useful?** An upvote helps it reach other people trying to learn
robotics simulation without a budget. Corrections and questions welcome below;
I answer all of them.

---

## Reproducing this

Every notebook in this series runs on free infrastructure. Nothing here needs
a paid GPU.

| Method | Free allowance | Best for |
|---|---|---|
| Kaggle | 30 GPU hr/week, 2x T4 | Running this notebook as-is |
| Lightning AI | 80 GPU hr/month, persistent disk | Heavy Isaac Sim generation |
| Google Colab | best-effort T4 | Quick smoke tests |
| NVIDIA DLI | free hosted labs | Learning the Isaac Sim GUI |

**The one gotcha worth remembering:** Isaac Sim needs **RT cores**. A T4, L4,
L40S or any RTX card is fine. An **A100 or H100 is not** — those have no RT
cores, so the RTX renderer is unsupported or unusably slow. It is the most
counterintuitive constraint in cloud robotics simulation, and it bites people
who assume the more expensive GPU must be the better one.

*Series index and full source: see the linked dataset description.*